# 03 — Build Feature Table
# Giai đoạn 1 — Mục 1.4 — Xây dựng bảng đặc trưng 32 chiều cho MLP

**Đầu ra:**
- `outputs/tables/features_mlp_squarelaw.parquet`
- `outputs/tables/features_mlp_hilbert.parquet`
- `outputs/tables/features_mlp_hybrid.parquet`
- `outputs/tables/features_mlp.parquet` (alias Square-Law, dùng cho GĐ2)

In [10]:
from pathlib import Path
import pandas as pd
import json
from fractions import Fraction
from scipy.signal import resample_poly

from common import io_utils, features_full, config as cfg

# Đã bỏ: numpy (không dùng), dsp (không còn gọi trực tiếp sau khi gộp vào
# features_full.build_full_feature_table), NOMINAL_RPM_BY_LOAD và
# SKF_6205_GEOMETRY (build_full_feature_table tự tra RPM từ config, khai lại
# ở notebook là mở đường cho hai nguồn số lệch nhau).

In [11]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Đọc manifest và cấu hình

In [12]:
manifest_clean = pd.read_csv(TABLES_DIR / "manifest_clean.csv")

with open(TABLES_DIR / "bandpass_config.json") as f:
    bandpass_cfg = json.load(f)

BAND_HZ     = tuple(bandpass_cfg["band_hz"])
LP_CUTOFF_HZ = bandpass_cfg["lp_cutoff_hz"]
TARGET_FS   = cfg.SCOPE["sampling_rate_hz"]
WINDOW_SIZE = cfg.WINDOW_SIZE_RAW
STRIDE      = cfg.STRIDE_RAW
WARMUP      = cfg.WARMUP_SAMPLES   # bỏ cửa sổ chạm quá độ đầu tín hiệu

# 'resolved_sample_rate_hz' là fs đã được io_utils.run_sanity_checks() chốt;
# cột 'fs' chỉ là giá trị khai báo ban đầu. build_full_feature_table() đọc
# cột resolved, nên notebook phải in và thử đúng cột đó.
FS_COL = "resolved_sample_rate_hz" if "resolved_sample_rate_hz" in manifest_clean.columns else "fs"

print(f"Số file trong manifest: {len(manifest_clean)}")
print(f"Bandpass: {BAND_HZ} Hz, LP cutoff: {LP_CUTOFF_HZ} Hz")
print(f"Target FS: {TARGET_FS} Hz, Window: {WINDOW_SIZE}, Stride: {STRIDE}, Warmup: {WARMUP} mẫu")
print(f"Phân phối sampling rate gốc (cột {FS_COL}):")
print(manifest_clean[FS_COL].value_counts().to_string())

Số file trong manifest: 40
Bandpass: (2300, 3800) Hz, LP cutoff: 750.0 Hz
Target FS: 12000 Hz, Window: 2048, Stride: 1024, Warmup: 2048 mẫu
Phân phối sampling rate gốc (cột resolved_sample_rate_hz):
resolved_sample_rate_hz
12000.0    36
48000.0     3
24000.0     1


## 2. Hàm load + resample tường minh

Resample xảy ra TẠI ĐÂY — không ẩn trong `build_full_feature_table()`.
File `.mat` trong `data/clean/` vẫn giữ nguyên fs gốc.
Resample chỉ xảy ra trong RAM khi đọc lên.

In [13]:
def load_and_resample(file_path, source_fs, target_fs=TARGET_FS):
    """
    Đọc tín hiệu DE từ file .mat rồi resample về target_fs.
    Sửa code ở: io_utils.load_de_signal_resampled().
    """
    return io_utils.load_de_signal_resampled(Path(file_path), source_fs, target_fs)


# Kiểm tra nhanh với file Normal (có thể khác 12kHz)
normal_rows = manifest_clean[manifest_clean["label"] == "Normal"]
for _, row in normal_rows.iterrows():
    x_test = load_and_resample(row["file_path"], row[FS_COL])
    print(f"File Normal: {Path(row['file_path']).name}")
    print(f"  fs gốc : {row[FS_COL]} Hz")
    print(f"  Sau resample: {len(x_test)} mẫu @ {TARGET_FS} Hz")
    print(f"  Thời lượng  : {len(x_test)/TARGET_FS:.2f} s")
    print()

File Normal: 100_Normal_3.mat
  fs gốc : 48000.0 Hz
  Sau resample: 121411 mẫu @ 12000 Hz
  Thời lượng  : 10.12 s

File Normal: 97_Normal_0.mat
  fs gốc : 24000.0 Hz
  Sau resample: 121969 mẫu @ 12000 Hz
  Thời lượng  : 10.16 s

File Normal: 98_Normal_1.mat
  fs gốc : 48000.0 Hz
  Sau resample: 120976 mẫu @ 12000 Hz
  Thời lượng  : 10.08 s

File Normal: 99_Normal_2.mat
  fs gốc : 48000.0 Hz
  Sau resample: 120976 mẫu @ 12000 Hz
  Thời lượng  : 10.08 s



## 3. Hàm build bảng đặc trưng cho 1 DSP method

Envelope được tính **1 lần / file** trên toàn bộ tín hiệu,
sau đó mới cắt sliding window — tránh reset bộ lọc IIR causal ở mỗi window.

In [14]:
def build_features(method_canonical):
    """Bảng đặc trưng 28 chiều cho toàn manifest với một phương pháp envelope.

    method_canonical: "square_law" | "hilbert_fir" | "hybrid"
        (tên chuẩn trong dsp.BENCHMARK_ENVELOPE_METHODS)

    load_and_resample vẫn được truyền vào để bước đọc + resample tường minh
    trong notebook, nhưng phần cắt cửa sổ / đặt file_id / trích đặc trưng thì
    dùng đúng một cài đặt duy nhất trong common/features_full.py.
    """
    return features_full.build_full_feature_table(
        manifest_clean,
        band_hz=BAND_HZ,
        load_de_signal_fn=load_and_resample,
        window_size=WINDOW_SIZE,
        stride=STRIDE,
        warmup_samples=WARMUP,
        target_fs_hz=TARGET_FS,
        lp_cutoff_hz=LP_CUTOFF_HZ,
        envelope_method=method_canonical,
        # ĐƯỜNG BAO GỐC: giữ đúng đầu ra mặc định của từng phương pháp, KHÔNG
        # bật take_sqrt. Square-Law vì vậy trả LP(x²) (đơn vị bình phương của
        # x) còn hilbert_fir/hybrid trả |A|. Chênh lệch đơn vị đó là ĐẶC TÍNH
        # của phương pháp, thuộc thứ RQ3 cần đo, không phải thứ cần triệt.
        #
        # HAI HỆ QUẢ PHẢI XỬ LÝ Ở GIAI ĐOẠN 2 (không xử lý bằng cách sửa DSP):
        #   1. Bộ chuẩn hóa đặc trưng phải được FIT RIÊNG trên tập train của
        #      TỪNG bảng. Dùng chung một bộ tham số chuẩn hóa cho cả 3 bảng thì
        #      nhánh squarelaw thua vì lệch thang, không phải vì chất lượng
        #      đường bao -> RQ3 đo sai.
        #   2. Lượng tử hóa INT8 (mục 2.3) nhạy với DẢI ĐỘNG, mà LP(x²) có dải
        #      động rộng hơn |A| -> báo cáo sai số INT8 riêng cho từng phương
        #      pháp và coi đó là KẾT QUẢ so sánh, không phải nhiễu.
        envelope_kwargs=None,
    )

## 4. Build 3 bảng đặc trưng cho 3 DSP method

> ⏱️ Mỗi method mất vài phút tùy số file và cấu hình máy.

In [15]:
# Tên nhánh (dùng cho tên file đầu ra)  ->  tên phương pháp chuẩn trong dsp.py
DSP_METHODS = {
    "squarelaw": "square_law",   # nhân quả, rẻ nhất cho MCU
    "hilbert":   "hilbert_fir",  # FIR Hilbert Type III 65 tap, NHÂN QUẢ
    "hybrid":    "hybrid",       # sai phân trung tâm + Alpha-Max Beta-Min
}
# 11 time + 10 order + 7 envelope. Order/envelope KHÔNG phải 12/9: ở độ phân
# giải 5.86 Hz/bin của cửa sổ 2048 mẫu, BPFO_h2~BSF_h3 và BPFO_h3~BPFI_h2
# không tách được. Để nguyên 2 cột thì chúng trùng nhau ở tải này nhưng
# khác nhau ở tải khác -> chính việc đó mã hóa TẢI, đúng biến LOLO đang đo.
# order.py gộp mỗi cặp thành 1 cột cố định cho mọi tải.
N_FEATURES_EXPECTED = 28

key_sets = {}
result_summary = []

for out_name, method in DSP_METHODS.items():
    print("-" * 62)
    print(f"Đang build đặc trưng - nhánh {out_name} (envelope_method={method})")
    print("-" * 62)

    df = build_features(method)
    df["dsp_method"] = out_name

    # --- CHỐT LẠI LỖI RÒ RỈ DỮ LIỆU ------------------------------------
    # Mỗi file_id phải gom NHIỀU cửa sổ. Nếu min == 1 thì file_id lại đang bị
    # gắn chỉ số cửa sổ và File-based Split sẽ rò rỉ y như bản cũ. Để assert
    # ở đây vì đây là chỗ duy nhất sinh ra khóa chia tập.
    win_per_file = df.groupby("file_id")["window_idx"].nunique()
    assert win_per_file.min() > 1, (
        "file_id đang bị gắn chỉ số cửa sổ, File-based Split sẽ rò rỉ. "
        f"file_id chỉ có 1 cửa sổ: {win_per_file[win_per_file == 1].head().to_dict()}"
    )
    assert df.groupby(["file_id", "window_idx"]).size().max() == 1, (
        "Trùng cặp (file_id, window_idx): một cửa sổ bị ghi hai lần."
    )
    
    key_sets[out_name] = set(zip(df["file_id"], df["window_idx"]))
    
    assert df["start_idx"].min() >= WARMUP, (
        "Cửa sổ chạm vùng quá độ đầu tín hiệu chưa bị loại - "
        "common/features_full.py chưa được cập nhật."
    )

    # Cột đặc trưng lấy theo HỢP ĐỒNG trong features_full, không liệt kê tay:
    # bảng thêm cột metadata mới cũng không âm thầm lọt vào X khi huấn luyện
    # (bug cũ: window_idx và start_idx bị tính thành đặc trưng, 28 -> 30 chiều).
    feature_cols = features_full.feature_columns(df, expected_count=N_FEATURES_EXPECTED)

    out_path = TABLES_DIR / f"features_mlp_{out_name}.parquet"
    df.to_parquet(out_path)

    result_summary.append({
        "method": out_name,
        "envelope_method": method,
        "n_rows": len(df),
        "n_files": int(df["file_id"].nunique()),
        "n_features": len(feature_cols),
        "win_min": int(win_per_file.min()),
        "win_max": int(win_per_file.max()),
        "labels": df["label"].value_counts().to_dict(),
        "output": out_path.name,
    })

    print(f"OK {out_name}: {len(df)} cửa sổ / {df['file_id'].nunique()} file gốc, "
          f"{len(feature_cols)} đặc trưng -> {out_path.name}")
    print(df["label"].value_counts().to_string())
    
    # Đối chiếu 3 bảng có thẳng hàng theo (file_id, window_idx)
    # Về lý thuyết việc cắt cửa sổ tất định (cùng manifest, window_size, stride,
    # warmup) nên không phụ thuộc phương pháp DSP. Assert ở đây để RQ3 và notebook
    # 05 (ghép cặp cửa sổ CNN) không âm thầm so sánh sai cửa sổ nếu giả định này
    # từng bị vi phạm do sửa code sau này.
    ref_name, ref_keys = next(iter(key_sets.items()))
    for name, keys in key_sets.items():
        assert keys == ref_keys, (
            f"Bảng '{name}' và '{ref_name}' không cùng tập (file_id, window_idx) - "
            "3 nhánh DSP không còn thẳng hàng, RQ3/notebook 05 sẽ so sánh sai cửa sổ."
    )
    print(f"OK: cả {len(key_sets)} bảng đều có cùng {len(ref_keys)} cặp (file_id, window_idx).")

--------------------------------------------------------------
Đang build đặc trưng - nhánh squarelaw (envelope_method=square_law)
--------------------------------------------------------------


OK squarelaw: 4623 cửa sổ / 40 file gốc, 28 đặc trưng -> features_mlp_squarelaw.parquet
label
OR        1389
B         1387
IR        1386
Normal     461
OK: cả 1 bảng đều có cùng 4623 cặp (file_id, window_idx).
--------------------------------------------------------------
Đang build đặc trưng - nhánh hilbert (envelope_method=hilbert_fir)
--------------------------------------------------------------
OK hilbert: 4623 cửa sổ / 40 file gốc, 28 đặc trưng -> features_mlp_hilbert.parquet
label
OR        1389
B         1387
IR        1386
Normal     461
OK: cả 2 bảng đều có cùng 4623 cặp (file_id, window_idx).
--------------------------------------------------------------
Đang build đặc trưng - nhánh hybrid (envelope_method=hybrid)
--------------------------------------------------------------
OK hybrid: 4623 cửa sổ / 40 file gốc, 28 đặc trưng -> features_mlp_hybrid.parquet
label
OR        1389
B         1387
IR        1386
Normal     461
OK: cả 3 bảng đều có cùng 4623 cặp (file_id, window_

## 5. Tạo alias và kiểm tra kết quả

In [16]:
import shutil

# Alias: features_mlp.parquet = nhánh Square-Law (mặc định cho giai đoạn 2)
src = TABLES_DIR / "features_mlp_squarelaw.parquet"
dst = TABLES_DIR / "features_mlp.parquet"
shutil.copy2(src, dst)
print("Alias: features_mlp.parquet <- features_mlp_squarelaw.parquet")

print("=" * 78)
print("TÓM TẮT KẾT QUẢ")
print("=" * 78)
for r in result_summary:
    print(f"  {r['method']:10s} | {r['n_rows']:>6} cửa sổ | {r['n_files']:>3} file gốc "
          f"| {r['n_features']} đặc trưng | {r['win_min']}-{r['win_max']} cửa sổ/file "
          f"| {r['output']}")
print("=" * 78)

df_check = pd.read_parquet(TABLES_DIR / "features_mlp.parquet")
display(df_check.head(5))

# Xuất CSV để quan sát trực tiếp (Excel, pandas...) — dữ liệu giống hệt bản
# .parquet, chỉ đổi định dạng cho dễ mở/xem tay. KHÔNG phải nguồn dữ liệu
# chính dùng để huấn luyện ở Giai đoạn 2 (vẫn đọc từ .parquet).
for out_name in DSP_METHODS:
    src_path = TABLES_DIR / f"features_mlp_{out_name}.parquet"
    dst_path = TABLES_DIR / f"features_mlp_{out_name}.csv"
    pd.read_parquet(src_path).to_csv(dst_path, index=False)
    print(f"Đã xuất: {dst_path.name}  ({dst_path.stat().st_size / 1024:.1f} KB)")

# Alias CSV, tương ứng file alias .parquet ở cell trên
pd.read_parquet(TABLES_DIR / "features_mlp.parquet").to_csv(TABLES_DIR / "features_mlp.csv", index=False)
print("Đã xuất: features_mlp.csv (alias Square-Law)")

# Metadata và đặc trưng đều đọc từ hợp đồng features_full, không gõ tay lại.
meta_present = [c for c in features_full.METADATA_COLUMNS if c in df_check.columns]
print(f"Cột metadata (theo hợp đồng features_full): {meta_present}")
print(f"Số cột đặc trưng: {len(features_full.feature_columns(df_check, expected_count=N_FEATURES_EXPECTED))}")
print(f"Số file gốc: {df_check['file_id'].nunique()} | số cửa sổ: {len(df_check)}")
print(f"Ví dụ file_id: {df_check['file_id'].iloc[0]} (phải là tên file gốc, KHÔNG có _win_N)")

Alias: features_mlp.parquet <- features_mlp_squarelaw.parquet
TÓM TẮT KẾT QUẢ
  squarelaw  |   4623 cửa sổ |  40 file gốc | 28 đặc trưng | 115-117 cửa sổ/file | features_mlp_squarelaw.parquet
  hilbert    |   4623 cửa sổ |  40 file gốc | 28 đặc trưng | 115-117 cửa sổ/file | features_mlp_hilbert.parquet
  hybrid     |   4623 cửa sổ |  40 file gốc | 28 đặc trưng | 115-117 cửa sổ/file | features_mlp_hybrid.parquet


,file_id,window_idx,start_idx,label,class_label,load_hp,fault_diameter_mils,time_mean,time_std,time_rms,...,order_BPFO_h3_and_BPFI_h2,order_BPFI_h3,envelope_BSF_h1,envelope_BPFO_h1,envelope_BSF_h2,envelope_BPFI_h1,envelope_BPFO_h2_and_BSF_h3,envelope_BPFO_h3_and_BPFI_h2,envelope_BPFI_h3,dsp_method
0,118_0,2,2048,B,B_007,0,7.0,0.015280,0.138958,0.139796,...,0.001728,0.003051,0.001229,0.002622,0.003427,0.002419,0.005087,0.001614,0.002708,squarelaw
1,118_0,3,3072,B,B_007,0,7.0,0.015306,0.128595,0.129503,...,0.001172,0.000607,0.003335,0.003694,0.000689,0.001512,0.001308,0.002303,0.002329,squarelaw
2,118_0,4,4096,B,B_007,0,7.0,0.015259,0.132713,0.133587,...,0.000914,0.001095,0.004566,0.001688,0.001188,0.002128,0.000948,0.001323,0.002201,squarelaw
3,118_0,5,5120,B,B_007,0,7.0,0.015253,0.138287,0.139125,...,0.001212,0.001244,0.003078,0.001847,0.002592,0.003104,0.005494,0.001487,0.002605,squarelaw
4,118_0,6,6144,B,B_007,0,7.0,0.014885,0.139252,0.140045,...,0.000829,0.001144,0.001952,0.001478,0.002667,0.000942,0.005177,0.000128,0.002383,squarelaw


Đã xuất: features_mlp_squarelaw.csv  (2800.5 KB)
Đã xuất: features_mlp_hilbert.csv  (2779.8 KB)
Đã xuất: features_mlp_hybrid.csv  (2775.5 KB)
Đã xuất: features_mlp.csv (alias Square-Law)
Cột metadata (theo hợp đồng features_full): ['file_id', 'window_idx', 'start_idx', 'label', 'class_label', 'load_hp', 'fault_diameter_mils', 'dsp_method']
Số cột đặc trưng: 28
Số file gốc: 40 | số cửa sổ: 4623
Ví dụ file_id: 118_0 (phải là tên file gốc, KHÔNG có _win_N)
